In [1]:
import pathlib
import re

import scipy.io
import pandas as pd
import numpy as np

import torch

The directory containing the data

In [2]:
data_directory = pathlib.Path.cwd().parent / 'javi' / 'simulations'
assert data_directory.exists()

An, e.g, FM sample file (though this will also be valid for AM and *carrier*)

In [3]:
sample_file = next(data_directory.rglob('FM*.mat'))
sample_file

PosixPath('/home/manu/Sync/git/trapallada/javi/simulations/FM/FM_SNR3.162_Simul53.mat')

Metadata is extracted from the name of the file 

In [4]:
re_signal = re.compile(r'(FM|AM|carrier)_SNR([0-9]*\.?[0-9]+)_Simul(\d+).*\.mat')
re_signal.match(sample_file.name).groups()

('FM', '3.162', '53')

A noise sample file

In [5]:
sample_file = next(data_directory.rglob('noise*.mat'))
sample_file

PosixPath('/home/manu/Sync/git/trapallada/javi/simulations/noise/noise_color0.43625.mat')

Metadata is extracted from the name of the file 

In [6]:
re_noise = re.compile(r'noise_color([0-9]*\.?[0-9]+).*\.mat')
re_noise.match(sample_file.name).group(1)

'0.43625'

A function to pack a bunch of `jpg` and `mat` files into *torch* `tensor`s

In [9]:
def pack_dir(input_dir: str | pathlib.Path, output_file_stem: str | pathlib.Path) -> tuple[torch.Tensor, pd.DataFrame]:

    input_dir = pathlib.Path(input_dir)
    assert input_dir.exists()

    # for the sake of simplicity
    output_file_stem = pathlib.Path(output_file_stem)

    signal_files = list(input_dir.rglob('*.mat'))

    signals = []
    labels = []
    SNRs = []
    # ids = []
    file_names = []
    colors = []


    for f in signal_files:

        signals.append(scipy.io.loadmat(f)['y'])

        if m:=re_signal.match(f.name):
            # label, SNR, id = m.groups()
            label, SNR, _ = m.groups()
            color = np.nan
        elif m:=re_noise.match(f.name):
            label, SNR = 'noise', np.nan
            color = m.group(1)
        else:
            print(f'Could not match {f.name}')
            raise ValueError

        labels.append(label)
        SNRs.append(float(SNR))
        # ids.append(int(id))
        file_names.append(f.relative_to(input_dir))
        colors.append(color)

    # results are packed in torch tensors (rather than numpy arrays)
    signals = torch.from_numpy(np.stack(signals))
    torch.save(signals, output_file_stem.with_stem(output_file_stem.stem + '_signals').with_suffix('.pt'))

    # df = pd.DataFrame({'label': labels, 'SNR': SNRs, 'id': ids, 'color': colors, 'file name': file_names})
    df = pd.DataFrame({'label': labels, 'SNR': SNRs, 'color': colors, 'file name': file_names})
    df['label'] = df['label'].astype('category')
    df['file name'] = df['file name'].astype('string')

    df.to_parquet(output_file_stem.with_stem(output_file_stem.stem + '_labels').with_suffix('.parquet'))

    return signals, df


In [10]:
signals, metadata_df = pack_dir(data_directory, '4_classes_modulated')
metadata_df.head()

,label,SNR,color,file name
0,FM,3.162,NaN,FM/FM_SNR3.162_Simul53.mat
1,FM,10.000,NaN,FM/FM_SNR10_Simul450.mat
2,FM,3.162,NaN,FM/FM_SNR3.162_Simul245.mat
3,FM,31.620,NaN,FM/FM_SNR31.62_Simul30.mat
4,FM,31.620,NaN,FM/FM_SNR31.62_Simul947.mat


In [15]:
signals.shape

torch.Size([12983, 1, 10000])

Minimum and maximum values are

In [25]:
# max_value, min_value = signals.max(), signals.min()
# max_value, min_value

tensor(233.9523, dtype=torch.float64)

and in *absolute* value

In [26]:
max_abs_value = signals.abs().max()
max_abs_value

tensor(233.9523, dtype=torch.float64)

Mean and standard devation:

In [27]:
signals.mean(axis=0), signals.std(axis=0)

(tensor([[ 0.6166,  0.6175,  0.6050,  ...,  0.0041,  0.0027, -0.0018]],
        dtype=torch.float64),
 tensor([[0.6650, 0.6826, 0.6918,  ..., 3.3032, 3.2960, 3.3038]],
        dtype=torch.float64))

In [29]:
# signals / max_abs_value